# 20 — scPoli atlas

Mirror of `10_train_atlas_scvi`: prefer the pipeline-trained model from
`methods_scpoli`; fall back to inline training using the HPO best params
if the pipeline artifact is missing.

**Open question (plan §17 Q4)**: confirm scPoli was trained with Target2
as a `condition_key` for prototype anchoring — if not, retraining with
the right condition_key may matter for the T+ vs T-_matched contrast.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))
from src.paths import MODEL_OUT, scenario_model_dir, scenario_input_parquet, scenario_optuna_csv

## Parameters

In [ ]:
QUERY_SOURCE = ""          # empty = use all sources; "source_8" = exclude that source
REFERENCE_SCENARIO = "scenario_5"
ATLAS_NAME = (
    f"scpoli_atlas_{REFERENCE_SCENARIO}_full"
    if not QUERY_SOURCE
    else f"scpoli_atlas_excl_{QUERY_SOURCE}"
)

## Path 1 — symlink the pipeline-trained model

In [ ]:
src_model_dir = scenario_model_dir(REFERENCE_SCENARIO, "scpoli")
dst = MODEL_OUT / ATLAS_NAME

if src_model_dir.exists():
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    dst.symlink_to(src_model_dir)
    print(f"symlinked {dst} -> {src_model_dir}")
else:
    print(f"NO pipeline model at {src_model_dir} — falling through to inline training")

## Path 2 — inline training (fallback)

In [ ]:
if not dst.exists():
    from scarches.models.scpoli import scPoli
    import pandas as pd
    from src.data_io import load_parquet_as_anndata, split_by_source

    SOURCE_COL = "Metadata_Source"
    BATCH_KEY = "Metadata_Source"
    LABEL_KEY = "Metadata_JCP2022"  # compound id; verify

    adata = load_parquet_as_anndata(scenario_input_parquet(REFERENCE_SCENARIO))
    ref = adata if not QUERY_SOURCE else split_by_source(adata, SOURCE_COL, QUERY_SOURCE)[0]

    params = pd.read_csv(scenario_optuna_csv(REFERENCE_SCENARIO, "scpoli"))
    p = params.sort_values("total", ascending=False).iloc[0].to_dict()
    hidden = [int(p[f"params_layer_{i}_size"]) for i in range(int(p["params_num_layers"]))]

    model = scPoli(
        adata=ref,
        condition_keys=[BATCH_KEY],
        cell_type_keys=LABEL_KEY,
        hidden_layer_sizes=hidden,
        latent_dim=int(p["params_latent_dim"]),
        embedding_dims=int(p["params_embedding_dims"]),
        recon_loss="mse",
    )
    model.train(
        n_epochs=600,
        pretraining_epochs=int(400 * float(p["params_pretrain_to_train_ratio"])),
        use_early_stopping=True,
        reload_best=True,
        alpha_epoch_anneal=int(p["params_alpha_epoch_anneal"]),
        eta=float(p["params_eta"]),
    )
    dst.mkdir(parents=True, exist_ok=True)
    model.save(str(dst), overwrite=True)
    print(f"inline-trained scPoli atlas to {dst}")

    # Save reference embedding so nb40 can build joint ref+query metrics
    import anndata as ad
    model.model.eval()
    import numpy as np
    ref_emb = model.get_latent(ref, mean=True)
    ref_out = ad.AnnData(obs=ref.obs[[BATCH_KEY, LABEL_KEY]])
    ref_out.obsm["X_scpoli_mapped"] = ref_emb
    ref_out.write_h5ad(dst / "reference.h5ad")
    print(f"saved reference embedding ({ref_emb.shape}) to {dst / 'reference.h5ad'}")